# 配电网可规划域

四节点五走廊基础算例与 case33 共用 Network 接口、四个模型类和以下三个流程函数。切换网架只修改参数单元中的 network；预算、候选线路和设备参数由 Network 提供。

三维图与 FR/MR 使用同一评价箱的网格中心。完整分类不等于连续边界精确；未获证点保留为灰色。

In [1]:
from pathlib import Path  # 管理本次实验输出目录。
from time import perf_counter  # 计量建模、查询与区域分类的总时间。
from hashlib import sha256  # 记录本次计算使用的源码版本。
import nbformat  # Notebook 指纹只包含源单元，排除运行输出。
import numpy as np  # 选型、割及区域标签的数组运算。
import pandas as pd  # 从原始结果生成展示表。
from threadpoolctl import threadpool_limits  # 各方法使用相同矩阵运算线程数。
from IPython.display import display, IFrame  # 展示 FR/MR 表及可旋转三维图。
from Network.case33bw import Case33  # 33 节点固定拓扑升级算例。
from Network.four_bus_five_corridor import FourBus  # 四节点五走廊的原始基础算例。
from model import PlanningEquations, PlanningModel, PlanningSP, ACPowerFlow  # 四个正式模型类。
from region import classify_orthant, split_grid_box  # 单调区域分类与索引盒二分。
from vertify import BenchmarkResult, METHODS, METHOD_NAMES  # 唯一结果文件与派生指标。
from plot import save_method_comparison  # 绘图只读取已计算结果。

In [2]:
network = Case33(candidate_count=16)  # 十六候选线路；改为 4/8 或 FourBus() 即切换基础算例。
BUDGETS = network.budgets  # 默认预算及费用单位由网架给出，也可在这里指定新的预算。
DIVISIONS = 8  # 每条负荷轴划分为 8 个等长单元，先完成十六候选的规模测试。
RECOMPUTE = False  # False 时读取本目录已有结果，不重新求解。
OUTPUT = Path('results')/network.name/f'planning_{len(network.line_options)}'  # 输出目录随网架和候选数量自动切换。

**联合割查询。** MP 选择逐线路型号和负荷，连续 SP 核验运行约束并给出 $a+b^Tp+d^Tx\geq0$。历史割在同一模型的查询间复用。固定负荷求最低投资；固定方向与预算求负荷边界。

In [3]:
def joint_benders(equations, *, power=None, budget=np.inf, direction=None, cuts=(), start=None, radial_gap_kw=1e-3):  # 同一个 MP/SP 循环服务投资与边界查询。
    problem = PlanningModel(equations,power=power,budget=budget,direction=direction,cuts_only=True)  # MP 只含负荷和逐线路选型。
    oracle, generated = PlanningSP(equations), []  # 连续 SP 共用物理方程，新割仅收集一次。
    bound = -np.inf if power is not None else np.inf  # 最小投资保留下界，最大负荷保留上界。
    with problem.model:  # 本次查询结束时释放主问题。
        for cut in cuts:  # 已有全局有效割可以跨负荷和预算复用。
            problem.add_cut(cut)  # 同时约束负荷 p 和建设变量 x。
        if start is not None:  # 混合方法将 LP 方案作为 MIP 初值。
            problem.x.Start = start  # 初值本身不是 SOCP 可行证书。
        while True:  # 每轮获得运行证书、有效分离割或明确的未确定状态。
            answer = problem.solve()  # 求当前联合割主问题的候选与全局界。
            if answer is None:  # 主问题已证不可行，则原规划查询也不可行。
                return None,generated  # 保留本次已产生的有效割供后续查询复用。
            bound = max(bound,answer['bound']) if power is not None else min(bound,answer['bound'])  # 累积最强有效全局界。
            answer['bound'] = bound  # 与最后的可行目标值分开保存。
            if answer['x'] is None:  # 没有整数候选时无法进行运行认证。
                return answer,generated  # 保留求解器的未确定状态，不判成域外。
            point = answer['p']  # 固定负荷查询必须检查原坐标。
            if power is None and point.sum()>0.:  # 射线查询允许规定的绝对边界间隙。
                point = point*max(0.,1.-radial_gap_kw/point.sum())  # 认证内缩点，同时保留原 MP 全局上界。
            checked = oracle.solve(answer['x'],point)  # 同一连续 SP 支持 LP 和 SOCP。
            if checked['feasible']:  # 原始物理约束通过后才能发布运行证书。
                value = answer['objective'] if power is not None else point.sum()  # 投资或实际认证负荷作为可行目标值。
                gap = value-bound if power is not None else bound-value  # 两类目标使用各自正确的界方向。
                tolerance = 1e-7 if power is not None else radial_gap_kw+1e-5  # 投资与负荷精度使用不同单位。
                answer.update(p=point,state=checked['state'],feasible=True,objective=value,  # 保存当前方案的真实运行证书。
                              status='optimal' if gap<=tolerance else 'feasible')  # 间隙未闭合时只称可行。
                return answer,generated  # 本次查询完成。
            if checked['cut'] is None:  # 没有证书也没有有效分离割时，结果仍未确定。
                answer.update(status='unknown',termination=checked['termination'],feasible=False,objective=None)  # 不把数值停滞解释为不可行。
                return answer,generated  # 外层按已有全局界分类，其余预算保留未知。
            generated.append(checked['cut'])  # 记录本次新增的全局有效割。
            problem.add_cut(checked['cut'])  # 收紧同一个 MP 后继续求解。

**独立 AC 参考。** SOCP 只负责搜索建设方案；完整 AC 等式负责认证。某个方案在当前负荷下已证 AC 不可行时，仅在该点排除该方案并继续搜索。未确定结果不形成排除约束。

In [4]:
def ac_planning_query(equations, power):  # 求固定负荷的 AC 可行建设方案及最低投资界。
    problem = PlanningModel(equations,power=power)  # 完整 MISOCP 提供候选方案和 AC 最低投资的下界。
    bound = -np.inf  # 尚未得到投资下界。
    with problem.model:  # 方案排除只属于当前负荷点，不进入跨点割池。
        while True:  # 按优化器给出的候选逐次进行独立 AC 认证。
            answer = problem.solve()  # 搜索尚未被证明 AC 不可行的选型。
            if answer is None:  # SOCP 外松弛已不可行，且此前排除均有 AC 不可行证据。
                return None  # 当前负荷不存在可行 AC 建设方案。
            bound = max(bound,answer['bound'])  # 保留全局有效的最小投资下界。
            answer['bound'] = bound  # 与 AC 可行方案的实际费用分别记录。
            if answer['x'] is None:  # 无整数候选时不能建立具体 AC 网架。
                return answer  # 保留未知及已有下界。
            choice = equations.choice(answer['x'])  # 按 Network 的项目顺序恢复型号编号。
            oracle = ACPowerFlow(equations.network.design(choice))  # AC 模型只读取这个方案的物理网架。
            try:  # 按需创建的非凸 AC 模型必须释放。
                status = int(oracle.classify(power)[0])  # 完整电流等式给出可行、不可行或未确定。
                if status==0:  # 不动点没有证书时才调用独立非凸 AC 求解。
                    status = oracle.global_status(power,None)  # 求解器未完成仍保持未确定。
            finally:  # 无论认证结论如何都释放当前方案资源。
                oracle.close()  # 未建立非凸模型时无需额外操作。
            if status==1:  # 只有独立 AC witness 才能认证参考区域。
                answer.update(feasible=True,status='optimal' if answer['objective']-bound<=1e-7 else 'feasible')  # 投资上下界闭合才称最优。
                return answer  # 这个可行费用可同时用于多个预算。
            if status==0:  # 没有不可行证明时，不允许排除这个选型。
                answer.update(status='unknown',termination='ac_unknown',feasible=False,objective=None)  # 未知不会被算作域外。
                return answer  # 保留已经得到的有效投资下界。
            problem.exclude(answer['x'])  # 仅排除本点已经证明 AC 不可行的建设向量。

**区域构造。** 对当前非负纯负荷径向网，低负荷方向保持可行：高角可行可认证整块，低角不可行可排除整块，其余块二分。一个最低投资查询同时分类全部预算。混合方法先执行自己的 LP 阶段，再用 LP 割与选型初值进行 SOCP 查询；LP 内点须重新认证。

In [5]:
def build_region(network, method, budgets, divisions, bounds):  # 任意候选数量共用同一个三维构域流程。
    assert np.all(network.r>0.) and np.all(network.reactance>=0.) and np.all(network.original_p>=0.) and np.all(network.original_q>=0.) and np.all(network.q_ratio>=0.) and np.all(network.vmax>=1.)  # 当前成块推断要求纯负荷径向单调性。
    bounds, budgets = np.asarray(bounds),np.asarray(budgets)  # 公共评价箱与预算使用统一数组。
    states = np.zeros((len(budgets),)+(divisions,)*3,dtype=np.int8)  # 1 域内、-1 域外、0 未确定。
    cuts, warm = [], {}  # 只保留推进求解所需的割池与混合阶段初值。
    phases = ('linear','socp') if method=='hybrid' else (method,)  # 混合方法独立承担自己的线性阶段。
    for phase in phases:  # 各阶段的证书只按对应物理模型解释。
        if method=='hybrid' and phase=='socp':  # LP 域外可以继承，LP 域内必须重新认证。
            states[states==1] = 0  # 不把无损模型的可行性当作 SOCP 可行性。
        equations = PlanningEquations(network,'socp' if phase=='ac' else phase)  # AC 用 SOCP 搜索，等式认证另行完成。
        visited = set()  # 同阶段同格点只查询一次，未知点也不反复重试。
        pending = [(j,np.zeros(3,dtype=int),np.full(3,divisions-1,dtype=int)) for j in reversed(range(len(budgets)))]  # 先处理低预算，再复用共享证据。
        while pending:  # 每次处理一个含网格中心的闭索引盒。
            j,lower,upper = pending.pop()  # 取出预算编号与待分类的区域块。
            block = (j,)+tuple(slice(int(a),int(b)+1) for a,b in zip(lower,upper))  # 转为三维数组切片。
            if np.all(states[block]!=0):  # 已被此前证书覆盖的块无需再次求解。
                continue  # 表面绘图顶点不会增加查询次数。
            for index in (upper,lower):  # 优先检查能认证整块的高角，再检查能排除整块的低角。
                key = tuple(map(int,index))  # 网格索引唯一确定物理负荷。
                if states[(j,)+key]!=0 or key in visited:  # 已知或已经尝试的点不重复求解。
                    continue  # 其余未知部分通过后续二分处理。
                power = (index+.5)*bounds/divisions  # 使用等体积网格的中心负荷，单位 kW。
                if phase=='ac':  # 参考域必须通过独立完整 AC 认证。
                    answer = ac_planning_query(equations,power)  # 搜索可行建设方案，而非只验证某一个固定方案。
                else:  # 三种近似方法共用同一联合割查询。
                    answer,new = joint_benders(equations,power=power,cuts=cuts,start=warm.get(key))  # LP 初值只用于混合 SOCP 阶段。
                    cuts.extend(new)  # 每条有效新割只加入一次，跨点复用。
                visited.add(key)  # 同阶段该点已完成一次实际查询。
                labels = np.zeros(len(budgets),dtype=np.int8)  # 先将所有预算保持未确定。
                if answer is None:  # 所有合法建设方案已被证明不可行。
                    labels[:] = -1  # 无限预算也不能使物理不可行点变为可行。
                else:  # 投资下界与可行费用分别决定域外和域内。
                    if answer['bound'] is not None:  # 只有有效全局下界才能排除某档预算。
                        labels[budgets<answer['bound']-1e-7] = -1  # 预算低于最小投资下界，故不可行。
                    if answer['feasible']:  # 一个真实运行证书就足以给出可行投资上界。
                        labels[budgets>=answer['objective']] = 1  # 足以支付该方案的预算均获得证书。
                        if method=='hybrid' and phase=='linear':  # 仅混合方法需要保留 LP 建设初值。
                            warm[key] = answer['x']  # 初值不代替下一阶段的运行认证。
                for k,status in enumerate(labels):  # 一个查询同时更新所有预算。
                    classify_orthant(states[k],index,status)  # 同一证书沿逐项负荷单调方向传播。
                if np.all(states[block]!=0):  # 当前块已完整分类。
                    break  # 无需再查询另一角。
            if np.any(states[block]==0) and np.any(upper>lower):  # 尚有未知且不是单个中心时继续细分。
                pending.extend((j,a,b) for a,b in split_grid_box(lower,upper))  # 两个子盒完整覆盖父盒且互不重叠。
    return states  # 仅返回唯一三态区域；不保存逐查询轨迹或方案副本。

**执行与保存。** 三个无预算 LP 轴向上界确定公共评价箱。随后直接运行四种方法，每个方法共享全部预算；只保存区域标签、方法总耗时和复现配置。

In [6]:
if RECOMPUTE:  # 从当前代码和网架重新求解。
    with threadpool_limits(limits=1):  # 四种方法统一采用单线程矩阵运算。
        prepared = perf_counter()  # 公共评价箱准备时间单独记录一次。
        equations, bounds = PlanningEquations(network,'linear'), []  # 无损模型为当前纯负荷 AC/SOCP 提供外界。
        for direction in np.eye(3):  # 分别求三个独立负荷轴的全局上界。
            problem = PlanningModel(equations,direction=direction)  # 无预算限制以覆盖全部预算的规划区域。
            with problem.model:  # 每个轴向查询及时释放优化器。
                answer = problem.solve()  # 使用完整紧凑 LP 规划模型。
            if answer is None or not np.isfinite(answer['bound']):  # 无有效有限外界时不能发布区域比较。
                raise RuntimeError('No finite planning bound for the common evaluation box')  # 明确说明初始化失败。
            bounds.append(min(network.power_limit,answer['bound']))  # 使用全局上界，不用未收敛的可行目标值截断区域。
        bounds = np.ceil(np.asarray(bounds)/10.)*10.  # 向外取整到 10 kW，保证包含各方法区域。
        metadata = dict(network=network.name,planning=True,cost_unit=network.cost_unit,load_nodes=list(network.load_nodes),  # 网架及坐标含义。
                        candidate_count=len(network.line_options),budgets=[None if np.isinf(b) else b for b in BUDGETS],  # 无限预算用 JSON null 表示。
                        divisions=DIVISIONS,bounds=bounds.tolist(),preparation_seconds=perf_counter()-prepared,seconds={})  # 配置和计时各保存一份。
        metadata['hashes'] = {name:sha256(Path(name).read_bytes()).hexdigest() for name in ('model.py','region.py','vertify.py',*network.sources)}  # 记录参与计算的物理模型源码指纹。
        metadata['hashes']['main.ipynb'] = sha256('\n'.join(c.source for c in nbformat.read('main.ipynb',4).cells).encode()).hexdigest()  # 排除运行输出，避免指纹自引用。
        result = BenchmarkResult(np.zeros((len(METHODS),len(BUDGETS))+(DIVISIONS,)*3,dtype=np.int8),metadata)  # 一份三态数组容纳四方法和全部预算。
        for method in ('linear','socp','hybrid','ac'):  # 各方法独立构域，混合方法计入自己的 LP 阶段。
            started = perf_counter()  # 方法时间包含建模、求解和网格分类。
            result.states[METHODS.index(method)] = build_region(network,method,BUDGETS,DIVISIONS,bounds)  # 直接调用主流程，不经过测试看门狗。
            metadata['seconds'][method] = perf_counter()-started  # 同一方法的全部预算只记录一个总耗时。
            print(f"{method}: {metadata['seconds'][method]:.2f} s",flush=True)  # 报告方法完成，便于观察运行进度。
    result.save(OUTPUT)  # 全部结果只写入一个 result.npz 文件。
else:  # 查看已有结果时不启动优化器。
    result = BenchmarkResult.load(OUTPUT)  # 图和指标均使用同一份原始数据。

**FR、MR 与时间。** $FR=|\widehat D\setminus D_{AC}|/|\widehat D|$，$MR=|D_{AC}\setminus\widehat D|/|D_{AC}|$。方法耗时覆盖其全部预算，不能将各预算行的重复显示相加；公共评价箱准备时间单独记录。

In [7]:
summary = pd.DataFrame(result.summary)  # 指标全部从唯一三态数组现场计算。
summary['method'] = summary['method'].map(dict(zip(METHODS,METHOD_NAMES)))  # 仅转换显示名称。
display(summary[['budget','method','fr_percent','mr_percent','total_seconds']].rename(columns={  # 显示用户关心的核心比较。
    'budget':f'预算 ({result.metadata["cost_unit"]})','method':'方法','fr_percent':'FR (%)',  # 标明预算单位。
    'mr_percent':'MR (%)','total_seconds':'整次方法耗时 (s)'}).style.format(precision=5))  # 未确定时单值为空，图中给出区间。

,预算 (相对投资单位),方法,FR (%),MR (%),整次方法耗时 (s)
0,0.00000,纯 SOCP 切割,0.00000,0.00000,791.78412
1,1.00000,纯 SOCP 切割,0.00000,0.00000,791.78412
2,2.00000,纯 SOCP 切割,0.00000,0.00000,791.78412
3,nan,纯 SOCP 切割,0.00000,0.00000,791.78412
4,0.00000,线性 + SOCP 精修,0.00000,0.00000,357.62200
5,1.00000,线性 + SOCP 精修,0.00000,0.00000,357.62200
6,2.00000,线性 + SOCP 精修,0.00000,0.00000,357.62200
7,nan,线性 + SOCP 精修,0.00000,0.00000,357.62200
8,0.00000,AC 数值参考,0.00000,0.00000,6.63811
9,1.00000,AC 数值参考,0.00000,0.00000,6.63811


In [8]:
page = save_method_comparison(result,OUTPUT)  # 红色遗漏、黄色多余、灰色未确定，四面板同步旋转。
display(IFrame(src=page.as_posix(),width='100%',height=1220))  # 读取已有三态网格绘图，不再触发求解。